In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Використання робочого навантаження

<span id="usage"></span>
Використання відображає споживання сервісу Qiskit Runtime і визначається часом блокування QPU для виконання робочих навантажень.

- Використання сесії вимірюється як час, що минув поки сесія залишається активною, оскільки ємність QPU резервується на весь час сесії, незалежно від того, чи виконуються робочі навантаження активно. Дивись [Тривалість сесії](/guides/run-jobs-session#session-length) для отримання додаткової інформації про переходи між станами сесії.
- Використання пакету вимірюється як сукупний час блокування QPU для виконання всіх завдань у пакеті.
- Використання одиночного завдання вимірюється як час блокування QPU для виконання цього завдання.

Зверни увагу, що провалені або скасовані завдання за певних обставин зараховуються до твого використання — дивись розділ [Провалені та скасовані завдання](#failed-job) для деталей.

Для користувачів Pay-As-You-Go Plan дивись [Управління витратами](/guides/manage-cost) для деталей щодо встановлення ліміту витрат.

<span id="failed-job"></span>
## Використання для провалених та скасованих завдань
Коли завдання провалене або скасоване, зафіксоване використання визначається таким чином:

* Режим завдання або пакету: якщо провал або скасування стались через системну помилку, зафіксоване використання дорівнює нулю. Для завдань, що провалились через помилку користувача, або коли користувач скасував завдання, зафіксоване використання відповідає будь-якому спожитому ресурсу до цього моменту, включаючи накладні витрати, понесені для підготовки QPU до виконання завдання.

* Режим сесії: зафіксоване використання — це астрономічний час активності сесії, незалежно від кількості завдань, що провалились або були скасовані.

<span id="view-usage"></span>
## Запит фактичного використання робочого навантаження
Після завершення робочого навантаження є кілька способів переглянути його фактичне використання:

- Запусти [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) або [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) у `qiskit-ibm-runtime` версії 0.30 або новішої. Якщо використовується старіша версія `qiskit-ibm-runtime` (>= 0.23 та < 0.30), використання можна знайти в `session.details()["usage_time"]` та `batch.details()["usage_time"]`.
- Використай [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails), щоб переглянути використання для конкретного пакету або сесії.
- Використай [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById), щоб переглянути використання для одного завдання.

<span id="instance-usage"></span>
## Перегляд використання екземпляра
Ти можеш переглянути використання екземпляра на сторінці [Instances](https://quantum.cloud.ibm.com/instances) або, для тих, хто має відповідні повноваження, на сторінці [Analytics](https://quantum.cloud.ibm.com/analytics). Зверни увагу, що сторінки можуть показувати різні показники використання, оскільки вони обчислюють його по-різному.

Сторінка Instances показує використання в реальному часі за останні 28 днів (ковзне вікно), до поточного часу поточного дня. Використання на сторінці Analytics перераховується щогодини та включає останні 28 повних днів; тобто показує використання від 00:00 28 днів тому до сьогодні, на початку поточної години.

## Оцінка використання перед відправкою завдання
Хоча отримати точну локальну оцінку складно через додаткові операції для придушення та пом'якшення помилок, ти можеш скористатись цією базовою формулою для наближеної оцінки використання:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` — це накладні витрати приблизно 2 секунди на підзавдання. Включає такі операції, як завантаження корисного навантаження в керуючу електроніку. Твоє завдання з примітивом може бути розбите на кілька підзавдань, якщо воно надто велике для одночасної обробки рушієм виконання.
- `rep_delay` — це [налаштовувана користувачем](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay) опція; її стандартне значення задається через `backend.default_rep_delay`, що на більшості Backend-ів IBM Quantum становить 250 мікросекунд. Зверни увагу, що зменшення `rep_delay` скорочує загальний час виконання на QPU, але ціною збільшення рівня помилок підготовки стану; дивись посібник [Виконання з динамічною частотою повторень](/guides/repetition-rate-execution) для отримання додаткової інформації.
- `<circuit length>` — це загальна довжина інструкцій. Кожна інструкція займає різну кількість часу на QPU, тому загальна довжина варіюється від Circuit до Circuit. Наприклад, вимірювання може займати в 56 разів більше часу, ніж Gate `x`. Для визначення точної тривалості кожної інструкції можна використовувати `backend.target[<instruction>][<qubit>].duration`. Типова довжина Circuit, ймовірно, становить від 50 до 100 мікросекунд. Якщо ти використовуєш техніки придушення або пом'якшення помилок з примітивами, до твого Circuit можуть бути додані додаткові інструкції, що збільшить загальну довжину Circuit.
    > **Note:** [Експериментальна опція `scheduler_timing`](/guides/visualize-circuit-timing) повертає загальний час Circuit, але це НЕ є часом, що використовується для виставлення рахунків.
- `<num executions>` — це загальна кількість Circuit, помножена на кількість вимірювань (shots), де Circuit — це ті, що генеруються після трансляції елементів PUB.
    - Якщо ти використовуєш техніки пом'якшення помилок з примітивами, у процесі пом'якшення можуть запускатись додаткові Circuit, що збільшить загальну кількість виконань. Крім того, просунуті техніки пом'якшення помилок, такі як PEA і PEC, мають значно вищі накладні витрати, оскільки вимагають запуску Circuit для навчання на шумі.
    - Estimator групує квантово-побітово комутуючі спостережувані, що зменшує кількість виконань.

Якщо ти не використовуєш жодних просунутих технік пом'якшення помилок або власного `rep_delay`, для швидкої оцінки можна скористатись формулою `2+0.00035*<num executions>`.

### Оцінка використання локально за допомогою Qiskit
Цей приклад коду демонструє, як використовувати Qiskit для обчислення часу Circuit: